# Assignment 1 - Exercise Setup
This notebook contains the necessary code setup for the accompanying exercises. 

# 1. Coordinate Descent

In [ ]:
def argmin_x1(x):
    """Your code here"""

def argmin_x2(x):
    """Your code here"""

def argmin_x3(x):
    """Your code here"""

def f(x):
    """Your code here"""

def coordinate_descent(f, argmin, x0, max_iter=100, verbose=False):
    """Your code here"""
    for i in range(max_iter):
        # And here
        pass

# 2. Gradient Descent

In [ ]:
def f(x):
    """Your code here"""

def gradient_descent(f, grad_f, eta, u0, v0, max_iter=100) -> tuple[list, list]:
    """Your code here"""

def eta_const(t,c=1e-3) -> float:
    """Your code here"""

def eta_sqrt(t,c=1e-3) -> float:
    """Your code here"""

def eta_multistep(t, milestones=[20, 50], c=1e-4, eta_init=1e-3) -> float:
    """Your code here"""

# 3. Polynomial Regression

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

# Load the dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame
# The description of this dataset applies to the data provided over Canvas
print(housing.DESCR)

In [ ]:
import pandas as pd

# Adapt path
df = pd.read_csv("california_housing_with_split.csv")

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

X_train = train.drop(columns=["MedHouseVal", "split"])
y_train = train["MedHouseVal"]

X_test = test.drop(columns=["MedHouseVal", "split"])
y_test = test["MedHouseVal"]

# 4. Deicsion trees

In [16]:
def gini_impurity(y):
    """
    Compute Gini impurity for a binary/multiclass label vector.
    """
    if len(y) == 0:
        return 0.0

    probs = y.value_counts(normalize=True)
    return 1.0 - np.sum(probs ** 2)

def information_gain(l, l_0, l_1):
    """
    Compute information gain using Gini impurity.
    """

    parent_impurity = gini_impurity(l)

    n = len(l)
    n_0 = len(l_0)
    n_1 = len(l_1)

    weighted_child_impurity = (
        (n_0 / n) * gini_impurity(l_0)
        + (n_1 / n) * gini_impurity(l_1)
    )

    return parent_impurity - weighted_child_impurity

def split(data, features, target, continuous_features):
    """
    Implements Algorithm 9: Split(D).

    Parameters
    ----------
    data : pd.DataFrame
        Dataset D.
    features : list[str]
        Feature names to consider.
    target : str
        Target column name.
    continuous_features : set[str]
        Features treated as continuous.

    Returns
    -------
    best_q : dict
        Description of the best decision function q.
    best_L0 : pd.DataFrame
        Subset where q(x) = 1.
    best_L1 : pd.DataFrame
        Subset where q(x) = 0.
    max_ig : float
        Best information gain.
    """

    max_ig = 0.0
    best_q = None
    best_L0 = None
    best_L1 = None

    for feature in features:

        # Compute split only on observed values for this feature
        D_feature = data[[feature, target]].dropna()

        if len(D_feature) == 0:
            continue

        is_continuous = feature in continuous_features
        thresholds = get_thresholds(D_feature[feature], is_continuous)

        for t in thresholds:

            if is_continuous:
                L0 = D_feature[D_feature[feature] >= t]
                L1 = D_feature[D_feature[feature] < t]

                q = {
                    "feature": feature,
                    "threshold": t,
                    "type": "continuous",
                    "rule": f"{feature} >= {t}"
                }

            else:
                L0 = D_feature[D_feature[feature] == t]
                L1 = D_feature[D_feature[feature] != t]

                q = {
                    "feature": feature,
                    "threshold": t,
                    "type": "discrete",
                    "rule": f"{feature} == {t}"
                }

            ig = information_gain(
                D_feature[target],
                L0[target],
                L1[target]
            )

            if ig > max_ig:
                max_ig = ig
                best_q = q
                best_L0 = L0
                best_L1 = L1

    return best_q, best_L0, best_L1, max_ig

def class_probability_vector(data, target):
    """
    Return empirical class probability vector p_D(y).
    """
    probs = data[target].value_counts(normalize=True).sort_index()
    return probs.to_dict()


def stopping_criterion(data, target, depth, max_depth=3, min_samples_split=2):
    """
    Basic CART stopping criterion.
    """
    if depth >= max_depth:
        return True

    if len(data) < min_samples_split:
        return True

    if data[target].nunique() == 1:
        return True

    return False


def CART(data, features, target, continuous_features, depth=0, max_depth=3):
    """
    Implements Algorithm 8: CART(D).

    Returns either:
      - a leaf node with class probabilities
      - an internal decision node q with CART(L0) and CART(L1)
    """

    if stopping_criterion(data, target, depth, max_depth):
        return {
            "type": "leaf",
            "probabilities": class_probability_vector(data, target),
            "n_samples": len(data)
        }

    q, L0, L1, max_ig = split(
        data=data,
        features=features,
        target=target,
        continuous_features=continuous_features
    )

    if q is None or max_ig <= 0 or len(L0) == 0 or len(L1) == 0:
        return {
            "type": "leaf",
            "probabilities": class_probability_vector(data, target),
            "n_samples": len(data)
        }

    return {
        "type": "decision",
        "q": q,
        "information_gain": max_ig,
        "n_samples": len(data),
        "L0": CART(
            L0,
            features,
            target,
            continuous_features,
            depth + 1,
            max_depth
        ),
        "L1": CART(
            L1,
            features,
            target,
            continuous_features,
            depth + 1,
            max_depth
        )
    }

def get_thresholds(series, continuous):
    values = sorted(series.dropna().unique())

    if continuous:
        return [
            (values[i] + values[i + 1]) / 2
            for i in range(len(values) - 1)
        ]

    return values


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/train.csv")

features = ["Sex", "Pclass", "Fare", "Age", "Embarked"]
continuous_features = {"Fare", "Age"}
target = "Survived"

In [18]:
D_sex = df[["Sex", "Survived"]].dropna()

L0 = D_sex[D_sex["Sex"] == "female"]
L1 = D_sex[D_sex["Sex"] != "female"]

ig_sex = information_gain(
    D_sex["Survived"],
    L0["Survived"],
    L1["Survived"]
)

print(ig_sex)

0.1396479574728524


In [19]:
q, L0, L1, max_ig = split(
    data=df,
    features=["Sex"],
    target="Survived",
    continuous_features=set()
)

print(q)
print(max_ig)

{'feature': 'Sex', 'threshold': 'female', 'type': 'discrete', 'rule': 'Sex == female'}
0.1396479574728524


In [20]:
q, L0, L1, ig = split(
    data=df,
    features=["Pclass"],
    target="Survived",
    continuous_features=set()
)

print(q)
print(ig)

{'feature': 'Pclass', 'threshold': np.int64(3), 'type': 'discrete', 'rule': 'Pclass == 3'}
0.049137852428292605


In [21]:
q, L0, L1, ig = split(
    data=df,
    features=["Fare"],
    target="Survived",
    continuous_features={"Fare"}
)

print(q)
print(ig)

{'feature': 'Fare', 'threshold': np.float64(10.48125), 'type': 'continuous', 'rule': 'Fare >= 10.48125'}
0.04258355157593807


In [22]:
q, L0, L1, ig = split(
    data=df,
    features=["Age"],
    target="Survived",
    continuous_features={"Age"}
)

print(q)
print(ig)

{'feature': 'Age', 'threshold': np.float64(6.5), 'type': 'continuous', 'rule': 'Age >= 6.5'}
0.0123447785042467


In [23]:
q, L0, L1, ig = split(
    data=df,
    features=["Embarked"],
    target="Survived",
    continuous_features=set()
)

print(q)
print(ig)

{'feature': 'Embarked', 'threshold': 'C', 'type': 'discrete', 'rule': 'Embarked == C'}
0.013645883939281234


# 5. Naive Bayes

In [5]:
from sklearn.datasets import fetch_20newsgroups
categories = ['sci.space', 'misc.forsale', 'comp.graphics', 'rec.sport.hockey']
train = fetch_20newsgroups(subset='train', categories=categories)
test = fetch_20newsgroups(subset='test', categories=categories)

print(train.DESCR)

.. _20newsgroups_dataset:

The 20 newsgroups text dataset
------------------------------

The 20 newsgroups dataset comprises around 18000 newsgroups posts on
20 topics split in two subsets: one for training (or development)
and the other one for testing (or for performance evaluation). The split
between the train and test set is based upon a messages posted before
and after a specific date.

This module contains two loaders. The first one,
:func:`sklearn.datasets.fetch_20newsgroups`,
returns a list of the raw texts that can be fed to text feature
extractors such as :class:`~sklearn.feature_extraction.text.CountVectorizer`
with custom parameters so as to extract feature vectors.
The second one, :func:`sklearn.datasets.fetch_20newsgroups_vectorized`,
returns ready-to-use features, i.e., it is not necessary to use a feature
extractor.

**Data Set Characteristics:**

=================   ==========
Classes                     20
Samples total            18846
Dimensionality               1

The classes are indicated categorically with indices from zero to two by the target vector. The target names tell us which index belongs to which class.

In [6]:
y_train = train.target
y_train

array([3, 3, 1, ..., 2, 1, 0], shape=(2362,))

In [26]:
train.target_names

['comp.graphics', 'misc.forsale', 'rec.sport.hockey', 'sci.space']

We represent the documents in a bag of word format. That is, we create a data matrix ``D`` such that ``D[j,i]=1`` if the j-th document contains the i-th feature (word), and ``D[j,i]=0`` otherwise. 

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
vectorizer = CountVectorizer(stop_words="english", min_df=5,token_pattern=r"[^\W\d_]+", binary=True)
D = vectorizer.fit_transform(train.data)
D_test = vectorizer.transform(test.data)

We get the allocation of feature indices to words by the following array, containing the vocabulary.

In [28]:
vectorizer.get_feature_names_out()

array(['aa', 'aargh', 'ab', ..., 'zubov', 'zv', 'zyeh'],
      shape=(7386,), dtype=object)

For example, the word `zubov` has the index 7383.

In [29]:
np.where(vectorizer.get_feature_names_out() == 'zubov')[0]

array([7383])

In [8]:
import numpy as np

alpha = 1e-5

vocab = vectorizer.get_feature_names_out()
chip_idx = np.where(vocab == "chip")[0][0]

for c in range(4):
    D_c = D[y_train == c]

    numerator = np.sum(D_c[:, chip_idx] == 1) + alpha
    denominator = D_c.shape[0] + alpha * 2

    log_prob = np.log(numerator / denominator)

    print(f"log p(x_chip = 1 | y = {c}) = {log_prob}")

log p(x_chip = 1 | y = 0) = -4.172675328628088
log p(x_chip = 1 | y = 1) = -4.0690257884263445
log p(x_chip = 1 | y = 2) = -4.6051685526561466
log p(x_chip = 1 | y = 3) = -6.385184432774537


In [9]:
import numpy as np

def posterior_probability(word, target_class, D, y_train, vectorizer, alpha=1e-5):
    """
    Compute:
        p(y = target_class | x_word = 1)

    using Bayes' theorem + Laplace smoothing.

    Parameters
    ----------
    word : str
        Vocabulary word.

    target_class : int
        Class index.

    D : sparse matrix or ndarray
        Binary bag-of-words matrix.

    y_train : ndarray
        Class labels.

    vectorizer : CountVectorizer
        Fitted sklearn vectorizer.

    alpha : float
        Laplace smoothing parameter.

    Returns
    -------
    float
        Posterior probability.
    """

    vocab = vectorizer.get_feature_names_out()

    if word not in vocab:
        raise ValueError(f"'{word}' not in vocabulary")

    word_idx = np.where(vocab == word)[0][0]

    classes = np.unique(y_train)

    priors = []
    likelihoods = []

    for c in classes:

        D_c = D[y_train == c]

        # prior p(y=c)
        prior = D_c.shape[0] / D.shape[0]
        priors.append(prior)

        # likelihood p(x_word=1 | y=c)
        numerator = np.sum(D_c[:, word_idx] == 1) + alpha

        # binary feature -> |X_k| = 2
        denominator = D_c.shape[0] + alpha * 2

        likelihood = numerator / denominator
        likelihoods.append(likelihood)

    priors = np.array(priors)
    likelihoods = np.array(likelihoods)

    numerator = (
        likelihoods[target_class]
        * priors[target_class]
    )

    denominator = np.sum(
        likelihoods * priors
    )

    posterior = numerator / denominator

    return posterior

In [10]:
print(posterior_probability(
    "electronics", 0, D, y_train, vectorizer
))

print(posterior_probability(
    "sale", 1, D, y_train, vectorizer
))

print(posterior_probability(
    "games", 2, D, y_train, vectorizer
))

print(posterior_probability(
    "ball", 3, D, y_train, vectorizer
))

0.21052639885139895
0.9573332578609131
0.7029702075058984
0.41666611111042245


# 6. Lasso vs Ridge regression

In [ ]:
# load data
import pandas as pd
df = pd.read_csv("data/METABRIC_RNA_Mutation.csv") #Adapt the path
df_D = pd.concat([df['age_at_diagnosis'], df.iloc[:, 31:520]],axis=1)
D = df_D.to_numpy()
y = df['overall_survival_months'].to_numpy()

/var/folders/ss/x8g9q17j5rs_bmj9nws5sn9r0000gn/T/ipykernel_99685/2913487456.py:3: DtypeWarning: Columns (0: rasgef1b_mut, 1: hras_mut, 2: smarcb1_mut, 3: siah1_mut) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("METABRIC_RNA_Mutation.csv") #Adapt the path


In [12]:
import numpy as np
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, mean_squared_error


def β_ridge_intercept(X, y, λ, intercept_id = 0):
    n, p= X.shape
    I = np.eye(p)
    I[intercept_id,intercept_id]=0
    return np.linalg.solve(X.T@X + 2*n*λ*I, X.T@y)

def β_lasso(X, y, λ, t_max=1000, intercept_id=0):
    n, p = X.shape
    β = np.zeros(p)

    for t in range(t_max):
        for k in range(p):
            # partial residual for coordinate k
            r_k = y - X @ β + X[:, k] * β[k]

            c_k = X[:, k].T @ r_k

            if k == intercept_id:
                β[k] = c_k / np.sum(X[:, k] ** 2)
            else:
                β[k] = (
                    np.sign(c_k)
                    * max(abs(c_k) - n * λ, 0)
                    / np.sum(X[:, k] ** 2)
                )

    return β


def lasso_cv_plot(X, y, lambdas, n_splits=5):
    train_mse_means = []
    test_mse_means = []
    selected_features_means = []

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for lam in lambdas:
        model = make_pipeline(
            StandardScaler(),
            Lasso(
                alpha=lam,
                fit_intercept=True,
                max_iter=10000,
                tol=1e-6
            )
        )

        cv_results = cross_validate(
            model,
            X,
            y,
            cv=kf,
            scoring="neg_mean_squared_error",
            return_train_score=True,
            return_estimator=True
        )

        train_mse = -cv_results["train_score"]
        test_mse = -cv_results["test_score"]

        train_mse_means.append(np.mean(train_mse))
        test_mse_means.append(np.mean(test_mse))

        selected_counts = []

        for estimator in cv_results["estimator"]:
            lasso = estimator.named_steps["lasso"]
            beta = lasso.coef_

            selected = np.sum(np.abs(beta) > 1e-16)
            selected_counts.append(selected)

        selected_features_means.append(np.mean(selected_counts))

    return (
        np.array(train_mse_means),
        np.array(test_mse_means),
        np.array(selected_features_means)
    )

X = D

lambdas = np.logspace(-10, 8, 100)


# beta = β_ridge_intercept(X, y, lambdas)
# f_x = X.T@beta

train_mse, test_mse, avg_selected = lasso_cv_plot(X, y, lambdas)

plt.figure(figsize=(8, 5))
plt.semilogx(lambdas, train_mse, label="Training MSE")
plt.semilogx(lambdas, test_mse, label="Test MSE")
plt.xlabel("λ")
plt.ylabel("5-fold CV MSE")
plt.title("Lasso: CV Training and Test MSE")
plt.legend()
plt.grid(True)
plt.show()


KeyboardInterrupt: 